In [ ]:
!pip install pdfplumber tokenizers pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.1/68.1 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 69.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 100.9 MB/s eta 0:00:00


In [ ]:
import pdfplumber

# ⚠️ Change this to your actual file path!
PDF_PATH = "/content/World_History_Volume_1.pdf"

raw_text_pages = []

with pdfplumber.open(PDF_PATH) as pdf:
    total_pages = len(pdf.pages)
    print(f"📖 Total pages: {total_pages}")

    for i, page in enumerate(pdf.pages):
        if i < 19:          # skip pages 1–17 (preface, contents, intro)
            continue
        text = page.extract_text()
        if text:            # skip blank pages
            raw_text_pages.append(text)

        if (i + 1) % 20 == 0:
            print(f"  ✅ {i+1}/{total_pages} pages done...")

# Join all pages into one big string
raw_text = "\n".join(raw_text_pages)

print(f"\n Extraction done!")
print(f" Total characters extracted: {len(raw_text):,}")
print(f"\n Sample (first 500 chars):")
print("-" * 50)
print(raw_text[:500])

📖 Total pages: 788
  ✅ 20/788 pages done...
  ✅ 40/788 pages done...
  ✅ 60/788 pages done...
  ✅ 80/788 pages done...
  ✅ 100/788 pages done...
  ✅ 120/788 pages done...
  ✅ 140/788 pages done...
  ✅ 160/788 pages done...
  ✅ 180/788 pages done...
  ✅ 200/788 pages done...
  ✅ 220/788 pages done...
  ✅ 240/788 pages done...
  ✅ 260/788 pages done...
  ✅ 280/788 pages done...
  ✅ 300/788 pages done...
  ✅ 320/788 pages done...
  ✅ 340/788 pages done...
  ✅ 360/788 pages done...
  ✅ 380/788 pages done...
  ✅ 400/788 pages done...
  ✅ 420/788 pages done...
  ✅ 440/788 pages done...
  ✅ 460/788 pages done...
  ✅ 480/788 pages done...
  ✅ 500/788 pages done...
  ✅ 520/788 pages done...
  ✅ 540/788 pages done...
  ✅ 560/788 pages done...
  ✅ 580/788 pages done...
  ✅ 600/788 pages done...
  ✅ 620/788 pages done...
  ✅ 640/788 pages done...
  ✅ 660/788 pages done...
  ✅ 680/788 pages done...
  ✅ 700/788 pages done...
  ✅ 720/788 pages done...
  ✅ 740/788 pages done...
  ✅ 760/788 pages done.

In [ ]:
import re

def clean_text(text):
    text = re.sub(r'http\S+|www\.\S+', '', text)           # remove URLs
    text = re.sub(r'[^a-zA-Z0-9.,!?\'\"\-\s]', ' ', text) # remove special chars
    text = re.sub(r'\b\d{1,4}\b', '', text)                # remove page numbers
    text = re.sub(r'\s+', ' ', text).strip()               # remove extra spaces
    text = text.lower()                                     # lowercase
    return text

cleaned_text = clean_text(raw_text)

# Before vs After — screenshot this for your PPT!
print(" BEFORE vs AFTER CLEANING")
print("=" * 50)
print(" BEFORE:")
print(raw_text[:300])
print("\n AFTER:")
print(cleaned_text[:300])
print("\n Size reduction:")
print(f"  Raw      : {len(raw_text):,} characters")
print(f"  Cleaned  : {len(cleaned_text):,} characters")
print(f"  Reduced  : {((len(raw_text)-len(cleaned_text))/len(raw_text)*100):.1f}%")

 BEFORE vs AFTER CLEANING
 BEFORE:
10 1 • Understanding the Past
1.1Developing a Global Perspective
LEARNING OBJECTIVES
By the end of this section, you will be able to:
• Identify the role history plays in higher education
• Discuss the ways in which thestudy of historycan build skills for lifelong learning and success
• Explain how 

 AFTER:
understanding the past .1developing a global perspective learning objectives by the end of this section, you will be able to identify the role history plays in higher education discuss the ways in which thestudy of historycan build skills for lifelong learning and success explain how the features of

 Size reduction:
  Raw      : 2,062,268 characters
  Cleaned  : 2,019,628 characters
  Reduced  : 2.1%


In [ ]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

# Save cleaned text
with open("history_1_cleaned.txt", "w", encoding="utf-8") as f:
    f.write(cleaned_text)



# Initialize tokenizer
tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
tokenizer.pre_tokenizer = Whitespace()

# Trainer
trainer = BpeTrainer(
    vocab_size=5000,
    min_frequency=2,
    special_tokens=["[UNK]", "[CLS]", "[SEP]", "[PAD]", "[MASK]"]
)

# Train
print("🚀 Training BPE tokenizer...")
tokenizer.train(files=["history_1_cleaned.txt"], trainer=trainer)

# Save tokenizer
tokenizer.save("history1_bpe_tokenizer.json")
print(" Tokenizer trained!")

sample_text = "learning objectives by the end of this section, you will be able to identify the role history plays in higher education discuss"
encoding = tokenizer.encode(sample_text)

print("\n🔤 Original Text:", sample_text)
print("🧩 Tokens:", encoding.tokens)
print("🔢 Token IDs:", encoding.ids)

🚀 Training BPE tokenizer...
 Tokenizer trained!

🔤 Original Text: learning objectives by the end of this section, you will be able to identify the role history plays in higher education discuss
🧩 Tokens: ['learning', 'objectives', 'by', 'the', 'end', 'of', 'this', 'section', ',', 'you', 'will', 'be', 'able', 'to', 'identify', 'the', 'role', 'history', 'play', 's', 'in', 'higher', 'education', 'discuss']
🔢 Token IDs: [848, 1893, 102, 49, 212, 58, 167, 1535, 8, 533, 809, 86, 338, 67, 2587, 49, 1172, 485, 1215, 40, 51, 3097, 2376, 1863]


In [ ]:
ex = tokenizer.encode(cleaned_text[:1000])

ex.tokens


['understanding',
 'the',
 'past',
 '.',
 '1',
 'developing',
 'a',
 'global',
 'perspective',
 'learning',
 'objectives',
 'by',
 'the',
 'end',
 'of',
 'this',
 'section',
 ',',
 'you',
 'will',
 'be',
 'able',
 'to',
 'identify',
 'the',
 'role',
 'history',
 'play',
 's',
 'in',
 'higher',
 'education',
 'discuss',
 'the',
 'ways',
 'in',
 'which',
 'the',
 'study',
 'of',
 'history',
 'can',
 'build',
 'skills',
 'for',
 'life',
 'long',
 'learning',
 'and',
 'success',
 'explain',
 'how',
 'the',
 'features',
 'of',
 'this',
 'text',
 'will',
 'optim',
 'ize',
 'your',
 'learning',
 'experience',
 'from',
 'the',
 'leg',
 'ends',
 'of',
 'tro',
 'y',
 'her',
 'al',
 'ded',
 'by',
 'hom',
 'er',
 'to',
 'the',
 'cont',
 'ents',
 'of',
 'dig',
 'ital',
 'arch',
 'ives',
 'access',
 'ed',
 'by',
 'modern',
 'students',
 ',',
 'the',
 'human',
 'story',
 'has',
 'f',
 'asc',
 'inated',
 'and',
 'instru',
 'cted',
 'those',
 'who',
 'have',
 'tried',
 'to',
 'understand',
 'its',
 'co

In [ ]:
ex.ids

[1843,
 49,
 948,
 10,
 12,
 4834,
 22,
 1934,
 3768,
 848,
 1893,
 102,
 49,
 212,
 58,
 167,
 1535,
 8,
 533,
 809,
 86,
 338,
 67,
 2587,
 49,
 1172,
 485,
 1215,
 40,
 51,
 3097,
 2376,
 1863,
 49,
 868,
 51,
 277,
 49,
 1399,
 58,
 485,
 528,
 2034,
 3461,
 92,
 543,
 570,
 848,
 61,
 725,
 1680,
 564,
 49,
 1979,
 58,
 167,
 1959,
 809,
 4324,
 872,
 1520,
 848,
 2512,
 138,
 49,
 760,
 2910,
 58,
 1867,
 46,
 381,
 62,
 1492,
 102,
 1119,
 54,
 67,
 49,
 355,
 633,
 58,
 3481,
 514,
 350,
 938,
 448,
 57,
 102,
 555,
 4022,
 8,
 49,
 490,
 1440,
 677,
 27,
 3213,
 3499,
 61,
 4766,
 519,
 653,
 217,
 268,
 2971,
 67,
 1313,
 226,
 1303,
 346,
 10,
 937,
 70,
 49,
 948,
 677,
 570,
 430,
 2105,
 22,
 2151,
 58,
 1005,
 8,
 61,
 226,
 1399,
 677,
 1547,
 430,
 320,
 631,
 10,
 81,
 268,
 156,
 131,
 907,
 49,
 1669,
 381,
 166,
 60,
 166,
 40,
 126,
 4617,
 40,
 3835,
 111,
 8,
 653,
 217,
 435,
 229,
 2743,
 138,
 485,
 196,
 435,
 71,
 57,
 67,
 4098,
 55,
 65,
 10,
 1646,
 747,

In [ ]:
import torch
import numpy as np
from transformers import BertTokenizer, BertModel


# ── Load pretrained BERT ──
print("⏳ Loading BERT model...")
bert_tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
bert_model     = BertModel.from_pretrained("bert-base-uncased")
bert_model.eval()
print("✅ BERT loaded!\n")

# ── Split book into chunks of 512 tokens ──
def chunk_text(text, chunk_size=512):
    """
    Splits the full book text into chunks of max 512 tokens
    because BERT has a 512 token limit.
    """
    # tokenize the whole text first
    tokens = bert_tokenizer.tokenize(text)
    print(f"📊 Total tokens in book : {len(tokens):,}")

    # split into chunks of 512
    chunks = []
    for i in range(0, len(tokens), chunk_size - 2):  # -2 for [CLS] and [SEP]
        chunk = tokens[i : i + chunk_size - 2]
        chunks.append(chunk)

    print(f"📦 Total chunks created : {len(chunks)}")
    return chunks

# ── Get BERT embeddings for one chunk ──
def get_chunk_embedding(token_chunk):
    """
    Takes a list of tokens, converts to IDs, passes through BERT,
    returns the embeddings for that chunk.
    """
    # convert tokens back to string for BERT input
    ids      = bert_tokenizer.convert_tokens_to_ids(token_chunk)
    ids      = [bert_tokenizer.cls_token_id] + ids + [bert_tokenizer.sep_token_id]
    id_tensor = torch.tensor([ids])

    with torch.no_grad():
        output     = bert_model(id_tensor)
        embeddings = output.last_hidden_state[0]  # shape: (chunk_len, 768)

    return embeddings.numpy()

# ── Run on full book ──
print("\n🚀 Processing full book...\n")

chunks     = chunk_text(cleaned_text)
all_embeddings = []

for i, chunk in enumerate(chunks):
    emb = get_chunk_embedding(chunk)
    all_embeddings.append(emb)

    # show progress every 10 chunks
    if (i + 1) % 10 == 0 or i == 0:
        print(f"  ✅ Chunk {i+1}/{len(chunks)} — shape: {emb.shape}")

# ── Stack all embeddings ──
# each chunk is (num_tokens, 768) — we stack them all
all_embeddings_combined = np.vstack(all_embeddings)

print(f"\n✅ Done!")
print(f"📐 Final embedding shape : {all_embeddings_combined.shape}")
print(f"   ↳ {all_embeddings_combined.shape[0]:,} total tokens × 768 dimensions")

# ── Save embeddings ──
np.save("history1_bert_embeddings.npy", all_embeddings_combined)
print(f"\n💾 Saved to history1_bert_embeddings.npy")

# ── Show sample output (screenshot for PPT!) ──
print("\n🔥 SAMPLE EMBEDDING (first token, first 8 values):")
print("-" * 50)
print(all_embeddings_combined[0][:8].round(4))
print("\n🔥 SAMPLE EMBEDDING (last token, first 8 values):")
print("-" * 50)
print(all_embeddings_combined[-1][:8].round(4))

⏳ Loading BERT model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ BERT loaded!


🚀 Processing full book...



Token indices sequence length is longer than the specified maximum sequence length for this model (409176 > 512). Running this sequence through the model will result in indexing errors


📊 Total tokens in book : 409,176
📦 Total chunks created : 803
  ✅ Chunk 1/803 — shape: (512, 768)
  ✅ Chunk 10/803 — shape: (512, 768)
  ✅ Chunk 20/803 — shape: (512, 768)
  ✅ Chunk 30/803 — shape: (512, 768)
  ✅ Chunk 40/803 — shape: (512, 768)
  ✅ Chunk 50/803 — shape: (512, 768)
  ✅ Chunk 60/803 — shape: (512, 768)
  ✅ Chunk 70/803 — shape: (512, 768)
  ✅ Chunk 80/803 — shape: (512, 768)
  ✅ Chunk 90/803 — shape: (512, 768)
  ✅ Chunk 100/803 — shape: (512, 768)
  ✅ Chunk 110/803 — shape: (512, 768)
  ✅ Chunk 120/803 — shape: (512, 768)
  ✅ Chunk 130/803 — shape: (512, 768)
  ✅ Chunk 140/803 — shape: (512, 768)
  ✅ Chunk 150/803 — shape: (512, 768)
  ✅ Chunk 160/803 — shape: (512, 768)
  ✅ Chunk 170/803 — shape: (512, 768)
  ✅ Chunk 180/803 — shape: (512, 768)
  ✅ Chunk 190/803 — shape: (512, 768)
  ✅ Chunk 200/803 — shape: (512, 768)
  ✅ Chunk 210/803 — shape: (512, 768)
  ✅ Chunk 220/803 — shape: (512, 768)
  ✅ Chunk 230/803 — shape: (512, 768)
  ✅ Chunk 240/803 — shape: (512, 768)

In [ ]:
import numpy as np
import torch
import math

# ── Load your saved BERT embeddings ──
print("⏳ Loading BERT embeddings...")
token_embeddings = torch.tensor(np.load("history1_bert_embeddings.npy"))
print(f"✅ Loaded! Shape: {token_embeddings.shape}")
print(f"   ↳ {token_embeddings.shape[0]:,} tokens × {token_embeddings.shape[1]} dims\n")

# ── Positional Encoding Function ──
def positional_encoding(seq_len, embed_dim):
    """
    Generates positional encoding for each token position.
    Uses sine and cosine functions — same method used in original
    'Attention is All You Need' paper (Transformer paper).

    For each position:
      PE(pos, 2i)   = sin(pos / 10000^(2i/embed_dim))
      PE(pos, 2i+1) = cos(pos / 10000^(2i/embed_dim))
    """
    pe = torch.zeros(seq_len, embed_dim)

    position = torch.arange(0, seq_len).unsqueeze(1).float()  # (seq_len, 1)
    div_term = torch.exp(
        torch.arange(0, embed_dim, 2).float() * -(math.log(10000.0) / embed_dim)
    )

    pe[:, 0::2] = torch.sin(position * div_term)  # even indices → sine
    pe[:, 1::2] = torch.cos(position * div_term)  # odd indices  → cosine

    return pe

# ── Generate Positional Encoding for full book ──
seq_len   = token_embeddings.shape[0]   # 248,460 tokens
embed_dim = token_embeddings.shape[1]   # 768 dims

print("🚀 Generating positional encodings...")
pos_encoding = positional_encoding(seq_len, embed_dim)
print(f"✅ Done! Positional encoding shape: {pos_encoding.shape}")

# ── Save positional encodings ──
np.save("history1_positional_encoding.npy", pos_encoding.numpy())
print(f"💾 Saved to history1_positional_encoding.npy")

# ── Show sample ──
print("\n🔥 SAMPLE POSITIONAL ENCODING RESULTS")
print("=" * 60)

for pos in [0, 1, 2, 10, 100]:
    print(f"\n📍 Position {pos} (first 8 values):")
    print(f"   {pos_encoding[pos][:8].numpy().round(4)}")

print("\n" + "=" * 60)
print(f"📊 Shape Summary:")
print(f"   Token Embeddings      : {token_embeddings.shape}")
print(f"   Positional Encoding   : {pos_encoding.shape}")
print("=" * 60)
print("👉 Next step → Input Embeddings = Token Emb + Positional Enc")

⏳ Loading BERT embeddings...
✅ Loaded! Shape: torch.Size([410782, 768])
   ↳ 410,782 tokens × 768 dims

🚀 Generating positional encodings...
✅ Done! Positional encoding shape: torch.Size([410782, 768])
💾 Saved to history1_positional_encoding.npy

🔥 SAMPLE POSITIONAL ENCODING RESULTS

📍 Position 0 (first 8 values):
   [0. 1. 0. 1. 0. 1. 0. 1.]

📍 Position 1 (first 8 values):
   [0.8415 0.5403 0.8284 0.5601 0.8153 0.5791 0.802  0.5974]

📍 Position 2 (first 8 values):
   [ 0.9093 -0.4161  0.928  -0.3726  0.9442 -0.3293  0.9581 -0.2863]

📍 Position 10 (first 8 values):
   [-0.544  -0.8391 -0.3318 -0.9433 -0.1066 -0.9943  0.1188 -0.9929]

📍 Position 100 (first 8 values):
   [-0.5064  0.8623 -0.2383 -0.9712  0.8764  0.4815 -0.9286  0.3711]

📊 Shape Summary:
   Token Embeddings      : torch.Size([410782, 768])
   Positional Encoding   : torch.Size([410782, 768])
👉 Next step → Input Embeddings = Token Emb + Positional Enc


In [ ]:
import numpy as np
import torch

# ── Load both saved files ──
print("⏳ Loading embeddings...")
token_embeddings = torch.tensor(np.load("history1_bert_embeddings.npy"))
pos_encoding     = torch.tensor(np.load("history1_positional_encoding.npy"))
print(f"✅ Token Embeddings shape     : {token_embeddings.shape}")
print(f"✅ Positional Encoding shape  : {pos_encoding.shape}")

# ── Final Step: Input Embeddings = Token Embeddings + Positional Encoding ──
print("\n🚀 Generating Input Embeddings...")
input_embeddings = token_embeddings + pos_encoding
print(f"✅ Done! Input Embeddings shape: {input_embeddings.shape}")

# ── Save final input embeddings ──
np.save("philosophy_input_embeddings.npy", input_embeddings.numpy())
print(f"💾 Saved to history1_input_embeddings.npy")

# ── Show sample (screenshot for PPT!) ──
print("\n🔥 FINAL INPUT EMBEDDINGS SAMPLE")
print("=" * 60)

sample_positions = [0, 1, 2]

for pos in sample_positions:
    print(f"\n📍 Token Position {pos}:")
    print(f"   🔵 Token Embedding       : {token_embeddings[pos][:6].numpy().round(4)}")
    print(f"   🟡 Positional Encoding   : {pos_encoding[pos][:6].numpy().round(4)}")
    print(f"   🟢 Input Embedding (sum) : {input_embeddings[pos][:6].numpy().round(4)}")

# ── Final Summary (screenshot this for your PPT!) ──
print("\n" + "=" * 60)
print("📊 COMPLETE PIPELINE SUMMARY")
print("=" * 60)
print(f"  📖 Book              : Introduction to Philosophy")
print(f"  🔢 Total Tokens      : {token_embeddings.shape[0]:,}")
print(f"  📐 Embedding Dims    : {token_embeddings.shape[1]}")
print(f"  📦 Files Saved       : 3")
print(f"     ├── history1_bert_embeddings.npy")
print(f"     ├── history1_positional_encoding.npy")
print(f"     └── history1_input_embeddings.npy")
print("=" * 60)
print("\n✅ PIPELINE COMPLETE!")
print("   PDF → Extraction → Cleaning → BPE → Token Embeddings")
print("   → Positional Encoding → Input Embeddings 🎉")

⏳ Loading embeddings...
✅ Token Embeddings shape     : torch.Size([410782, 768])
✅ Positional Encoding shape  : torch.Size([410782, 768])

🚀 Generating Input Embeddings...
✅ Done! Input Embeddings shape: torch.Size([410782, 768])
💾 Saved to history1_input_embeddings.npy

🔥 FINAL INPUT EMBEDDINGS SAMPLE

📍 Token Position 0:
   🔵 Token Embedding       : [ 0.1755 -0.0006 -0.3258  0.2001 -0.0865 -0.0504]
   🟡 Positional Encoding   : [0. 1. 0. 1. 0. 1.]
   🟢 Input Embedding (sum) : [ 0.1755  0.9994 -0.3258  1.2001 -0.0865  0.9496]

📍 Token Position 1:
   🔵 Token Embedding       : [-0.3197  0.5833 -0.5385  0.0146  0.7589  0.4436]
   🟡 Positional Encoding   : [0.8415 0.5403 0.8284 0.5601 0.8153 0.5791]
   🟢 Input Embedding (sum) : [0.5218 1.1236 0.2899 0.5747 1.5741 1.0227]

📍 Token Position 2:
   🔵 Token Embedding       : [-0.46   -0.1243  0.2881  0.0911  1.2612 -0.181 ]
   🟡 Positional Encoding   : [ 0.9093 -0.4161  0.928  -0.3726  0.9442 -0.3293]
   🟢 Input Embedding (sum) : [ 0.4493 -0.54